# Prepare CS1 Reports From PARN Download to Report Saving

In [3]:
print("\n\n#######################")
print("#   START r28_cs1.py  #")
print("#######################\n\n")



#######################
#   START r28_cs1.py  #
#######################




In [6]:
import time

start_time = time.time()
start_time_r28_cs1 = start_time

# libraries, libraries!
print("Importing libraries ...")
from datetime import datetime
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm, notebook
from constants import pthPy, pthTest, pth_dl, cs1_reporting_nb
from utilities import timediff, last_working_day, prior_month_end, osprey, batch_list, r_classifier
import subprocess

print(f"\n {timediff(start_time, time.time())} importing libraries\n")

Importing libraries ...

 0.0sec importing libraries



In [5]:
# get inputs to pass to osprey()
start_time = time.time()
print("Collecting input data ...")

# get report fund codes from the py_report.xlsm 'arc' sheet
funds = pd.read_excel(pthPy, sheet_name = "arc", usecols = "N").dropna()  # funds
# funds.iloc[:, 0] = funds.iloc[:, 0].str.upper()  # capitalise fund codes
funds.columns = ["Funds"] # rename column
funds = funds['Funds'].apply(str.upper) # capitalise fund codes, also converts dataframe to a series
print(funds, type(funds))

# get report date
df = pd.read_excel(pthPy, sheet_name="arc", usecols="S", nrows = 9)
k = df.iloc[1, 0]
rptDate = k.date() if k == k else prior_month_end(datetime.today()).date()
# prior month end or report date override; type is datetime()
print(rptDate, k)

# check inputs
s = "" if len(funds) == 1 else "s"
print(
    f"{len(funds)} PARN CS1 fund report{s} as at \
{rptDate.strftime('%A %d %b %Y')} to be \
downloaded:\n  {(',').join(funds)}"
)
print(f"\n {timediff(start_time, time.time())} collecting input data\n")

0    PPSBAL_C
Name: Funds, dtype: object <class 'pandas.core.series.Series'>
2026-01-31 nan
1 PARN CS1 fund report as at Saturday 31 Jan 2026 to be downloaded:
  PPSBAL_C

 3.1sec collecting input data

<class 'pandas.core.series.Series'>


In [6]:
# download the PARN reports in batches
start_time = time.time()
num_batches = 2 if len(funds) != 1 else 1
es = "es" if num_batches != 1 else ""
print(f"\nDownloading the {len(funds)} fund holdings \
for {rptDate.strftime('%a %d %b %Y')} in {num_batches} \
batch{es} ...")

batch_size = int(len(funds) / num_batches)
batches = batch_list(funds, batch_size = min(len(funds),batch_size))
batch_filepaths = []
for index, batch in tqdm(enumerate(batches, start = 1)):
    fln = f"{index}_of_{len(batches)}_CS1"
    filename = f"PARN {fln}({len(batch)}) {rptDate.strftime('%#d%b%Y')}.csv"
    print(f"\nGet {filename}, a batch of {len(batch)} files:\n   {(', ').join(batch)}\n")    
    batch_filepath = os.path.join(pth_dl, filename)
    batch_filepaths.append(batch_filepath)
    if os.path.isfile(batch_filepath):
        print(f"\n{batch_filepath} exists\n")
        pass
    else:
        print(f"\n Downloading batch {index} of {len(batches)} as {batch_filepath}...\n")
        osprey('parn', (',').join(batch), rptDate, rptDate, fln, 'csv')

print(f"{timediff(start_time, time.time())} downloading \
the {len(funds)} fund holdings for \
{rptDate.strftime('%a %d %b %Y')} in {num_batches} batches\n")

1it [00:00, 997.46it/s]


Get PARN 1_of_1_CS1(1) 31Jan2026.csv, a batch of 1 files:
   PPSBAL_C


C:\Users\hilton.netta\Downloads\PARN 1_of_1_CS1(1) 31Jan2026.csv exists

0.0sec downloading the 1 fund holdings for Sat 31 Jan 2026 in 1 batches



In [7]:
# join the downloaded holding reports into a dataframe
start_time = time.time()
print('Dataframing the fund holding reports for CS1 reporting\n')

holdings = pd.DataFrame()
for batch_filepath in batch_filepaths:
    # print(f" {batch_filepath}")
    df_new   = pd.read_csv(batch_filepath)
    holdings = pd.concat([holdings,df_new])

# convert date columns from type object
date_cols = ['Next Coupon Date', 'Maturity Date', 'i Position Effective Date']
for date_col in date_cols:
    holdings[date_col] = pd.to_datetime(holdings[date_col])

# convert value columns from type object to type float
value_cols = ['Original Nominal', 'Clean Book Value', 'Clean Market Value',
              'Accrued Income', 'Dividend Receivable', 'Sum of Market Value Income', 
              'Market Value %', 'Current Exposure']
for value_col in value_cols:
    holdings[value_col] = holdings[value_col].astype(str).str.replace(",","").astype(float)

print(f" {len(holdings['Entity Name'].unique())} fund holdings \
as at {holdings['i Position Effective Date'].iloc[0].strftime('%d %b %Y')} in the dataframe")

print(f'\n{timediff(start_time, time.time())} dataframing the fund holding reports for CS1 reporting\n')

Dataframing the fund holding reports for CS1 reporting

 1 fund holdings as at 31 Jan 2026 in the dataframe

0.0sec dataframing the fund holding reports for CS1 reporting



In [8]:
# get the fund NAVs with osprey()
start_time = time.time()
s = "s" if len(funds) != 1 else ""
d = "s'" if len(funds) != 1 else "'s" 
print(f"\nGetting the {len(funds)} CS1 fund{d} \
NAV{s} as at {rptDate.strftime('%A %d %B %Y')} \
with osprey() ...")

name = 'CS1'
navs_fln = os.path.join(pth_dl, f'FNAV {name}({len(funds)}) {rptDate.strftime("%d%b%Y")}.csv')

if os.path.exists(navs_fln):
    print(f"\n CS1 fund NAVs as at {rptDate.strftime('%a %d %b %Y')} exists:\n  {navs_fln}\n")
    pass
else:
    osprey('fnav', (',').join(funds), rptDate, rptDate, name, 'csv')

print(f" {timediff(start_time, time.time())} getting \
the {len(funds)} fund{d} NAV{s} \
as at {rptDate.strftime('%A %d %B %Y')} \
with osprey()\n")


Getting the 1 CS1 fund's NAV as at Saturday 31 January 2026 with osprey() ...

 CS1 fund NAVs as at Sat 31 Jan 2026 exists:
  C:\Users\hilton.netta\Downloads\FNAV CS1(1) 31Jan2026.csv

 0.0sec getting the 1 fund's NAV as at Saturday 31 January 2026 with osprey()



In [9]:
# dataframe the downloaded fund NAVs
start_time = time.time()
print(f"\nDataframing the fund NAV{s} ...")

# dataframe the NAVs
navs = pd.read_csv(navs_fln)

#convert totals column from type object to type float
navs['Total Net Assets'] = navs['Total Net Assets'].str.replace(',', '').astype('float64')

print(f"\n {timediff(start_time, time.time())} dataframing the fund NAV{s} ...\n")


Dataframing the fund NAV ...

 0.0sec dataframing the fund NAV ...



In [10]:
# merge the CS1 fund holdings and NAVs, and compare their totals 
start_time = time.time()
print(f"Merging and comparing the {len(funds)} CS1 fund holdings and NAVs as at {rptDate.strftime('%A %d %B %Y')} ...")

holdings_totals = holdings.groupby('Entity ID', as_index = False).sum(numeric_only = True)[['Entity ID','Sum of Market Value Income','Current Exposure']]
sums_cf         = holdings_totals.merge(navs, how = 'left', left_on = 'Entity ID', right_on = 'NAV Entity ID')
sums_cf.drop(['Entity Name', 'NAV Entity ID'], axis = 1, inplace = True)
sums_cf['SoMVI-CE']  = sums_cf['Sum of Market Value Income'] - sums_cf['Current Exposure']
sums_cf['1-CE/SoMVI %']  = (1 - sums_cf['Current Exposure']/sums_cf['Sum of Market Value Income']) * 100
sums_cf['SoMVI-NAV'] = abs(sums_cf['Sum of Market Value Income'] - sums_cf['Total Net Assets'])
sums_cf['1-NAV/SoMVI %']  = (1 - sums_cf['Total Net Assets']/sums_cf['Sum of Market Value Income']) * 100
sums_cf = sums_cf.sort_values(by='SoMVI-NAV', ascending=False)
cols_order = ['Effective Date','Entity ID','Sum of Market Value Income','Current Exposure','Total Net Assets',\
              'SoMVI-CE','1-CE/SoMVI %','SoMVI-NAV','1-NAV/SoMVI %']
sums_cf = sums_cf[cols_order]

# set the number of decimals to present
cols_2dp = ['Sum of Market Value Income','Current Exposure','Total Net Assets']
for col in cols_2dp:
    sums_cf[col] = sums_cf[col].apply(lambda x: f"{x:,.2f}")

cols_6dp = ['SoMVI-CE', '1-CE/SoMVI %','SoMVI-NAV','1-NAV/SoMVI %']
for col in cols_6dp:
    sums_cf[col] = sums_cf[col].apply(lambda x: f"{x:,.6f}")

# sums_cf
# ...

print(f" {timediff(start_time, time.time())} merging and comparing the {len(funds)} \
CS1 fund holdings and NAVs as at {rptDate.strftime('%A %d %B %Y')}")

Merging and comparing the 1 CS1 fund holdings and NAVs as at Saturday 31 January 2026 ...
 0.0sec merging and comparing the 1 CS1 fund holdings and NAVs as at Saturday 31 January 2026


In [11]:
# convert the PARN holdings into Reg 28 format with correspodning headings
start_time = time.time()
print(f"\nConverting the CS1 fund PARN holdings in readiness for Reg 28 classification ...")

s = "" if len(funds) == 1 else "s"
cs1_fname = os.path.join(pthTest, f'CS1 PARN holdings ({len(funds)}) {rptDate.strftime("%d%b%Y")}.xlsx')

hold_cols = ['Entity ID', 'Investment Type','i Issue Name','PrimaryAssetID','CCY','Sum of Market Value Income',\
                           '% of Total Market Value','Current Exposure']
hReg28 = holdings[hold_cols] # identify the subset of holdings columns to be used
hReg28 = hReg28.rename(columns={'Entity ID': 'Entity Name', 'PrimaryAssetID': 'Primary Asset ID', 'Sum of Market Value Income': 'End Market Value',
                                '% of Total Market Value': 'Percentage of Market Value', 'Current Exposure': 'Closing Exposure PA'})
hReg28.insert(5, 'Reg28 Classification','') # insert the classification column as the new column 5
hReg28.insert(9, f'{rptDate.strftime("%d %b %Y")}','') # insert the report date as a header in the last column
hReg28.iloc[0,9] = cs1_fname
hReg28.iloc[1,9] = "CS1"
hReg28.reset_index(drop=True, inplace=True)

# hReg28.info()

print(f" {timediff(start_time, time.time())} converting the CS1 fund PARN holdings in readiness for Reg 28 classification\n")


Converting the CS1 fund PARN holdings in readiness for Reg 28 classification ...
 0.0sec converting the CS1 fund PARN holdings in readiness for Reg 28 classification



In [12]:
# write the CS1 holdings dataframe to review it as a worksheet
start_time = time.time()
print("\nWriting the CS1 fund holdings dataframe and navs dataframe to a sheet ...")

writer = pd.ExcelWriter(cs1_fname, engine = 'xlsxwriter')     # instantiate a sheet writer
hReg28.to_excel(  writer, index = False, sheet_name = 'All')  # write the NAV sheet
holdings.to_excel(writer, index = False, sheet_name = 'PARN')  # write the NAV sheet
sums_cf.to_excel( writer, index = False, sheet_name = 'NAVs') # write the missing NAVs sheet
writer.close() # https://pandas.pydata.org/docs/reference/api/pandas.ExcelWriter.html   class for writing DataFrame objects into excel sheets
print(f' \n{cs1_fname}\n')

print(f" {timediff(start_time, time.time())} writing the CS1 fund holdings dataframe and navs dataframe to a sheet\n")


Writing the CS1 fund holdings dataframe and navs dataframe to a sheet ...
 
\\PIM-CPT-FS.prescient.local\PIM-Documents$\Working Folders\Hilton\W\Reg_Tests\CS1 PARN holdings (1) 31Jan2026.xlsx

 2.6sec writing the CS1 fund holdings dataframe and navs dataframe to a sheet



In [13]:
# run the CS1 Reg 28 classification script
start_time = time.time()
print(f"\nClassifying the no-lookthrough holdings for the CS1 reports ...")

r_classifier('cs1', cs1_fname,) # classification script

print(f" {timediff(start_time, time.time())} classifying the no-lookthrough holdings for the CS1 reports\n")


Classifying the no-lookthrough holdings for the CS1 reports ...
1min 50.9sec             executing issuers_1.ipynb
 1min 50.9sec classifying the no-lookthrough holdings for the CS1 reports



In [3]:
# run the CS1 reporting script
start_time = time.time()
print(f'Generating the CS1 reports ...\n')

# scrpt = pth_gitrepo + r"\cs1_reporting.ipynb"
%run "$cs1_reporting_nb"

print(f' \n{timediff(start_time, time.time())} generating the CS1 reports\n')

Generating the CS1 reports ...

Importing openpyxl and some of its functions
 0.0sec importing openpyxl and some of its functions 


Getting the CS1 report inputs ...

 Reg 28 CS1 report 
 Saturday 31 Jan 2026 
 ZAR/USD = 15.7354
 1 fund: 
  PPSBAL_C

 3.1sec: getting the CS1 report inputs completed


Setting constants ...
 0.4sec setting constants completed

 1 out of the 1 funds were classified with issuers_1.ipynb as at 31 January 2026.
  Funds not classified (0):
  <empty>


  0%|          | 0/1 [00:00<?, ?it/s]

 PPSBAL_C (excl CLNs): 0.5sec CS1 sheet, 0.3sec Reg28 sheet, \\PIM-CPT-FS.prescient.local\PIM-Documents$\Working Folders\Hilton\W\Reg_Tests\PPSBAL_C Reg28 CS1 Derivative Report 31Jan2026.xlsx


Funds with no holdings at Saturday 31 Jan 2026 (0):
 
Funds with no derivatives at Saturday 31 Jan 2026 (0): 

Combining workbook for 1 funds at 31 Jan 2026 ...

 0.4sec combining workbook for 1 funds at 31 Jan 2026

 1.4sec CS1 report roundtrip time for 1 funds at 31 Jan 2026

 
2.2sec generating the CS1 reports



In [4]:
print(f"\n{timediff(start_time_r28_cs1, time.time())} roundtripping download, merge and Reg 28 CS1 reports\n")


52.6sec roundtripping download, merge and Reg 28 CS1 reports



In [17]:
print('\n\n#####################')
print('# END r28_cs1.ipynb #')
print('#####################\n\n')



#####################
# END r28_cs1.ipynb #
#####################




In [5]:
# !jupyter nbconvert --to script cs1_r28.ipynb # convert from .ipynb to .py

This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    Execute the notebook prior to export.
    Equivalent to: [--ExecutePr

[NbConvertApp] WARNING | pattern 'cs1_r28.ipynb' matched no files
[NbConvertApp] WARNING | pattern '#' matched no files
[NbConvertApp] WARNING | pattern 'convert' matched no files
[NbConvertApp] WARNING | pattern 'from' matched no files
[NbConvertApp] WARNING | pattern '.ipynb' matched no files
[NbConvertApp] WARNING | pattern 'to' matched no files
[NbConvertApp] WARNING | pattern '.py' matched no files
